**SAPIANS** · SCC5819 · MÓDULO 00 — A BASE SRAG · AUDITORIA DA COMPARAÇÃO DE BALANCEAMENTO

# O que precisa conferir antes de escolher uma estratégia

O walkthrough explica o experimento; este caderno confere a amostra, as listas
de configurações, a seleção e a aritmética das métricas registradas. Não treina
um preditor nem apresenta a leitura do agregado como reprodução das previsões.

[README do módulo](../README.md) · [Protocolo](../COURSE_STUDY.md)

---

**Objetivos.** Separar teste de validação, verificar orçamento comparável entre
estratégias e distinguir o controle de hiperparâmetros históricos das previsões
publicadas anteriormente.


In [1]:
%pip install -q numpy pandas pyarrow scikit-learn matplotlib


Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib
import json
import os
import pathlib
import sys
import tempfile
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss

RANDOM_STATE = 42
REPO = pathlib.Path(os.environ.get("INTERPRETABLE_ML_REPO", pathlib.Path.cwd())).resolve()
while REPO != REPO.parent and not (REPO / "tools/sapians.py").exists():
    REPO = REPO.parent
if not (REPO / "tools/sapians.py").exists():
    raise RuntimeError("Abra o caderno em um checkout do repositório.")
sys.path.insert(0, str(REPO / "tools"))
WORK = pathlib.Path(tempfile.mkdtemp(prefix="srag-sample-balance-audit-"))
os.chdir(WORK)
import sapians as SP
SP.aplicar()
RECORD = REPO / "modules/00-dataset/gold/sample_balance_search.json"
r = json.loads(RECORD.read_text()) if RECORD.exists() else {"status": "pending"}
protocol = r.get("protocol", {})
trials = r.get("validation_candidates", [])
selection = r.get("selection")
evaluation = r.get("evaluation")
print("Estado do registro:", r["status"])
print("Este caderno não ajusta modelos nem escolhe pelo resultado do teste.")


Estado do registro: complete
Este caderno não ajusta modelos nem escolhe pelo resultado do teste.


## §1 — A mesma amostra, com avaliação temporal separada

A comparação recebe a amostra comprometida no repositório. Alterar estratégia
não autoriza alterar os registros de validação e teste. O hash identifica o
arquivo inteiro; as contagens e os anos verificam o papel de cada partição.


In [3]:
def sha_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for part in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(part)
    return digest.hexdigest()

def object_hash(value):
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()
    return hashlib.sha256(encoded).hexdigest()

if protocol:
    assert object_hash({k:v for k,v in protocol.items() if k != "protocol_sha256"}) == protocol["protocol_sha256"]
    sample = REPO / "modules/00-dataset/gold/gold_covid_obito_sample.parquet"
    assert sha_file(sample) == protocol["source"]["sha256"]
    cohort = pd.read_parquet(sample, columns=["split", "y_obito", "gold_id", "ano_onset"])
    assert cohort["gold_id"].is_unique
    assert cohort["y_obito"].isin([0,1]).all()
    split_rows=[]
    for split, lower, upper in (("train",2020,2022),("val",2023,2023),("test",2024,2024)):
        part = cohort.loc[cohort["split"].eq(split)]
        assert len(part) == protocol["splits"][split]["n"]
        assert part["ano_onset"].between(lower,upper).all()
        if "events" in protocol["splits"][split]:
            assert int(part["y_obito"].sum()) == protocol["splits"][split]["events"]
        split_rows.append({"partição":split,"registros":len(part),"óbitos":int(part["y_obito"].sum())})
    assert len(cohort.loc[cohort["split"].eq("train")]) == 200_000
    print(SP.tabela(pd.DataFrame(split_rows)))
    year_mix = cohort.loc[cohort["split"].eq("train")].groupby("ano_onset", observed=True).agg(registros=("y_obito","size"), obitos=("y_obito","sum")).reset_index()
    year_mix["ano_onset"] = year_mix["ano_onset"].astype(str)
    print("Composição da amostra de treino por ano de início dos sintomas:")
    print(SP.tabela(year_mix))
    print("Amostra aleatória uniforme do período de treino; não há cotas iguais por ano.")
    print("O tamanho do treino foi uma decisão operacional, não um cálculo de suficiência estatística.")
    print("Validação e teste: todos os registros admitidos pela receita em seus respectivos anos.")
    print("Hash da amostra corresponde ao protocolo:", True)
    print("Registros identificados sem sobreposição entre partições:", True)
else:
    print("PENDENTE — o protocolo da nova comparação ainda não foi promovido.")


partição registros óbitos
   train   200.000 62.719
     val    24.148  4.572
    test    16.142  2.945
Composição da amostra de treino por ano de início dos sintomas:
ano_onset registros obitos
     2020    65.610 21.087
     2021   116.302 36.380
     2022    18.088  5.252
Amostra aleatória uniforme do período de treino; não há cotas iguais por ano.
O tamanho do treino foi uma decisão operacional, não um cálculo de suficiência estatística.
Validação e teste: todos os registros admitidos pela receita em seus respectivos anos.
Hash da amostra corresponde ao protocolo: True
Registros identificados sem sobreposição entre partições: True


## §2 — Mesmas configurações candidatas, escolhas próprias

O protocolo deve oferecer a mesma lista de parâmetros do XGBoost a cada
estratégia. Isso não significa que cada uma escolherá a mesma configuração.
A proporção da reamostragem e a receita de pesos são decisões declaradas;
a busca não é uma otimização irrestrita de todos os parâmetros possíveis.


In [4]:
if protocol:
    specs = protocol["candidates"]
    assert len({row["candidate_id"] for row in specs}) == len(specs)
    strategies = list(dict.fromkeys(row["strategy"] for row in specs))
    assert set(strategies) == {"plain","weights","rus","smotenc_restricted"}
    normalized = {strategy: sorted(object_hash(row["params"]) for row in specs if row["strategy"]==strategy) for strategy in strategies}
    assert all(values == normalized["plain"] for values in normalized.values())
    assert all(row["family"] == "xgb" for row in specs)
    assert len(protocol["features"]) == 40 and len(set(protocol["features"])) == 40
    print(SP.tabela(pd.DataFrame([{"estratégia":key,"configurações":len(value)} for key,value in normalized.items()])))
    print("Configurações registradas:", SP.pt_int(len(specs)))
    print("Configurações concluídas:", SP.pt_int(len(trials)))
    print("Entradas brutas do preditor:", SP.pt_int(len(protocol["features"])))
    by_spec = {row["candidate_id"]:row for row in specs}
    for row in trials:
        expected = by_spec[row["candidate_id"]]
        assert all(row[key] == expected[key] for key in ("family","strategy","params"))
        assert row["protocol_sha256"] == protocol["protocol_sha256"]
        assert row["status"] == "complete"
    print("Configuração e protocolo de cada resultado concluído: conferem.")
else:
    print("PENDENTE — nenhum orçamento é presumido sem o protocolo.")


        estratégia configurações
             plain            30
           weights            30
               rus            30
smotenc_restricted            30
Configurações registradas: 120
Configurações concluídas: 120
Entradas brutas do preditor: 40
Configuração e protocolo de cada resultado concluído: conferem.


## §3 — Perda, contagens e calibração precisam ser coerentes

A fórmula abaixo confere o significado do log loss com probabilidades
inteiramente artificiais, sem interpretar a perda como percentual de erro.
Em seguida, conferimos a aritmética das métricas agregadas reais. Isso não
substitui a conferência das previsões salvas feita pelo exportador do estudo.


In [5]:
y_demo=np.array([0,1,0,1])
p_demo=np.array([0.1,0.8,0.2,0.7])
probability_of_observed=np.where(y_demo==1,p_demo,1-p_demo)
manual=float(-np.log(probability_of_observed).mean())
library=float(log_loss(y_demo,p_demo,labels=[0,1]))
np.testing.assert_allclose(manual,library,rtol=0,atol=1e-14)
print("Ensaio artificial: log loss manual =",SP.pt(manual,8))
print("Ensaio artificial: log loss da biblioteca =",SP.pt(library,8))
print("O ensaio acima não é uma medição SRAG.")

def audit_metrics(m):
    n=m["n"]; events=m["events"]
    assert n>0 and 0<events<n and m["logloss"]>=0
    assert all(0<=m[k]<=1 for k in ("brier","auc","ap","mean_predicted","observed"))
    np.testing.assert_allclose(m["observed"],events/n,rtol=0,atol=1e-12)
    np.testing.assert_allclose(m["calibration_gap"],m["mean_predicted"]-m["observed"],rtol=0,atol=1e-12)
    bins=m["reliability"]
    assert sum(b["n"] for b in bins)==n
    assert len({b["bin"] for b in bins})==len(bins)
    for b in bins:
        assert b["n"]>0 and 0<=b["mean_predicted"]<=1 and 0<=b["observed"]<=1
    np.testing.assert_allclose(sum(b["n"]*b["mean_predicted"] for b in bins)/n,m["mean_predicted"],rtol=0,atol=1e-12)
    np.testing.assert_allclose(sum(b["n"]*b["observed"] for b in bins)/n,m["observed"],rtol=0,atol=1e-12)
    ece=sum(b["n"]*abs(b["mean_predicted"]-b["observed"]) for b in bins)/n
    np.testing.assert_allclose(ece,m["ece_10_equal_width"],rtol=0,atol=1e-12)
    assert m["tp"]+m["fn"]==events
    assert m["tn"]+m["fp"]==n-events
    np.testing.assert_allclose(m["sensitivity"],m["tp"]/(m["tp"]+m["fn"]),rtol=0,atol=1e-12)
    np.testing.assert_allclose(m["specificity"],m["tn"]/(m["tn"]+m["fp"]),rtol=0,atol=1e-12)

for row in trials:
    audit_metrics(row["validation"])
    assert row["validation"]["n"]==protocol["splits"]["val"]["n"]
print("Registros de validação com aritmética conferida:",SP.pt_int(len(trials)))


Ensaio artificial: log loss manual = 0,22708064
Ensaio artificial: log loss da biblioteca = 0,22708064
O ensaio acima não é uma medição SRAG.
Registros de validação com aritmética conferida: 120


## §4 — Congelar pela validação antes de observar o teste

Cada finalista precisa ser o mínimo do seu braço. O vencedor global precisa
ser o mínimo de todos os candidatos registrados, sem arredondar a perda.
O controle conserva os hiperparâmetros históricos, mas foi reexecutado com o
pré-processamento deste protocolo; sua previsão não é presumida idêntica à
publicada anteriormente.


In [6]:
if selection:
    assert object_hash({k:v for k,v in selection.items() if k!="selection_sha256"}) == selection["selection_sha256"]
    assert selection["protocol_sha256"]==protocol["protocol_sha256"]
    assert len(trials)==len(protocol["candidates"])
    assert selection["test_used_for_selection"] is False
    by_id={row["candidate_id"]:row for row in trials}
    for final in selection["finalists"]:
        same=[row for row in trials if row["strategy"]==final["strategy"]]
        expected=min(same,key=lambda row:(row["validation"]["logloss"],row["candidate_id"]))
        assert final["candidate_id"]==expected["candidate_id"]
        assert final["params"]==expected["params"]
    expected=min(trials,key=lambda row:(row["validation"]["logloss"],row["candidate_id"]))
    assert selection["winner"]["candidate_id"]==expected["candidate_id"]
    assert selection["control"]["candidate_id"]==protocol["control_candidate_id"]
    assert selection["control"]["strategy"]=="plain"
    print("Cada finalista minimiza o log loss de seu braço:",True)
    print("Vencedor minimiza o log loss de validação sem arredondamento:",True)
    print("O teste foi usado para selecionar:",False)
    print("Candidato vencedor:",selection["winner"]["candidate_id"])
else:
    print("PENDENTE — a escolha exige todos os candidatos concluídos.")

if evaluation:
    assert selection and evaluation["selection_sha256"]==selection["selection_sha256"]
    assert evaluation["selection_changed"] is False
    assert evaluation["canonical_model"]==selection["canonical_model"]
    candidates={row["candidate_id"] for row in selection["finalists"]}
    assert {row["candidate_id"] for row in evaluation["finalists"]}==candidates
    evaluated=list(evaluation["finalists"])+[evaluation["control"]]
    for row in evaluated:
        audit_metrics(row["test"])
        assert row["test"]["n"]==protocol["splits"]["test"]["n"]
    audit_metrics(evaluation["test"])
    print("Teste do vencedor e dos finalistas: contagens e calibração conferem.")
    print("A escolha foi alterada depois do teste:",False)
    print("Avaliação confirmatória independente:",False)
else:
    print("PENDENTE — nenhuma conclusão de teste antes de congelar a seleção.")


Cada finalista minimiza o log loss de seu braço: True
Vencedor minimiza o log loss de validação sem arredondamento: True
O teste foi usado para selecionar: False
Candidato vencedor: 7dcb4ad1cb16bb2d
Teste do vencedor e dos finalistas: contagens e calibração conferem.
A escolha foi alterada depois do teste: False
Avaliação confirmatória independente: False


## §5 — O que esta auditoria não demonstra

A leitura de um agregado, mesmo íntegro, não valida clinicamente as probabilidades.
O exportador confere os arquivos de previsões locais antes da promoção; este
caderno confere o registro público, o protocolo e as relações entre suas métricas.
Não calcula de novo as previsões sem o preditor salvo e as entradas correspondentes.

A busca é finita. A receita de pesos, a razão da reamostragem e a semente estão
fixadas para isolar a comparação declarada; os parâmetros do XGBoost recebem
ajustes próprios. Não houve recalibração de probabilidades nesta rodada.
O teste histórico já foi observado e sua reutilização é exploratória.
